# Medical Necessity — Source Data Validation

Profiles `temp_tnet_tripmaster` before it is used for medical necessity analysis. Nothing here
scores an order or calls a model. The purpose is to establish what each column actually
contains, which column really holds the clinical justification, how much of the population
survives the scope filters, and how much of the knowledge base vocabulary the data supports.

The nurse navigation work established the failure mode this notebook is built to catch: a field
that looked numeric turned out to be a disposition taxonomy, and a derived value agreed with
itself for five months before anyone checked. Every column here is treated as unknown until
profiled.

## 1. Configuration

In [ ]:
import os
import re
import json
import shutil
from datetime import datetime

import pandas as pd
import numpy as np
from pyspark.sql import functions as SF
from pyspark.sql import types as ST

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

SOURCE_TABLE = "`prod-sandbox`.vivekkumar_patel.temp_tnet_tripmaster"
KNOWLEDGE_PATH = "/Workspace/Users/josh.smitherman@gmr.net/med_nec/med_nec_knowledge.json"
OUTPUT_DIR = "/Workspace/Users/josh.smitherman@gmr.net/med_nec/data"
LOCAL_DIR = "/tmp"

SAMPLE_N = 20000
SAMPLE_SEED = 42

EXCLUDED_SERVICE_CODES = ["EMG", "WC", "FWQUOTE", "ORGAN"]
MIN_TEXT_CHARS = 20

INCLUDE_TEXT_SAMPLES = False
LOW_CARDINALITY_MAX = 60
TOP_VALUES = 25

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

print("run:", RUN_TS)
print("source:", SOURCE_TABLE)
print("text samples in output:", INCLUDE_TEXT_SAMPLES)

## 2. Table inventory

Row count, column count, and declared types. A column typed as string that holds only a few
distinct values is a category, not free text, regardless of what its name suggests.

In [ ]:
src = spark.table(SOURCE_TABLE)
ROW_COUNT = src.count()

schema_frame = pd.DataFrame(
    [{"column": f.name, "spark_type": f.dataType.simpleString(), "nullable": f.nullable}
     for f in src.schema.fields])

STRING_COLS = [f.name for f in src.schema.fields if isinstance(f.dataType, ST.StringType)]
NUMERIC_COLS = [f.name for f in src.schema.fields
                if isinstance(f.dataType, (ST.IntegerType, ST.LongType, ST.DoubleType,
                                           ST.FloatType, ST.DecimalType, ST.ShortType))]
DATE_COLS = [f.name for f in src.schema.fields
             if isinstance(f.dataType, (ST.DateType, ST.TimestampType))]

print("rows:", f"{ROW_COUNT:,}")
print("columns:", len(src.columns),
      "| string:", len(STRING_COLS),
      "| numeric:", len(NUMERIC_COLS),
      "| date:", len(DATE_COLS))
print()
print(schema_frame.to_string(index=False))

## 3. Column resolution audit

`find_col` is the same resolver the review notebook uses. It is run here with every candidate
printed so a wrong match is visible before 500 model calls are spent on the wrong field. Set
the overrides by hand in the cell below if any role resolves incorrectly or not at all.

In [ ]:
def find_col(df, candidates, contains=None):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    if contains:
        for lc, orig in cols.items():
            if all(tok in lc for tok in contains):
                return orig
    return None


ROLE_SPEC = {
    "order_id": (["order_id", "trip_id", "transport_id"], ["order", "id"]),
    "notes": (["medical_necessity_notes", "clinical_notes", "reason_for_transport",
               "justification", "notes"], ["reason"]),
    "facility": (["facility_name", "origin_facility", "pickup_facility"], ["facility"]),
    "service_level": (["service_level", "level_of_service", "service_code"], ["service"]),
    "created": (["created_date", "order_date", "transport_date"], ["date"]),
    "payer": (["payer", "payer_name", "insurance"], ["payer"]),
}

resolution = []
for role, (cands, contains) in ROLE_SPEC.items():
    picked = find_col(src, cands, contains)
    alternates = [c for c in src.columns
                  if any(tok in c.lower() for tok in [role.split("_")[0]] + [x.lower() for x in cands])]
    resolution.append({
        "role": role,
        "resolved": picked,
        "n_alternates": len(alternates),
        "alternates": ", ".join(sorted(set(alternates))[:6]),
    })

resolution_frame = pd.DataFrame(resolution)
print(resolution_frame.to_string(index=False))
print()

OVERRIDES = {}

COLS = {r["role"]: OVERRIDES.get(r["role"], r["resolved"]) for r in resolution}
unresolved = [k for k, v in COLS.items() if v is None]
print("active mapping:", COLS)
print("UNRESOLVED, set in OVERRIDES before continuing:" if unresolved else "all roles resolved",
      unresolved if unresolved else "")

## 4. Completeness

Null and blank are counted separately. A column that is technically populated with empty
strings will pass a null check and fail everything downstream.

In [ ]:
agg = []
for c in src.columns:
    col = SF.col(f"`{c}`")
    if c in STRING_COLS:
        agg.append(SF.sum(SF.when(col.isNull(), 1).otherwise(0)).alias(f"{c}__null"))
        agg.append(SF.sum(SF.when(SF.length(SF.trim(col)) == 0, 1).otherwise(0)).alias(f"{c}__blank"))
    else:
        agg.append(SF.sum(SF.when(col.isNull(), 1).otherwise(0)).alias(f"{c}__null"))
        agg.append(SF.lit(0).alias(f"{c}__blank"))

counts = src.agg(*agg).toPandas().iloc[0].to_dict()

completeness = []
for c in src.columns:
    n_null = int(counts[f"{c}__null"])
    n_blank = int(counts[f"{c}__blank"])
    populated = ROW_COUNT - n_null - n_blank
    completeness.append({
        "column": c,
        "populated": populated,
        "null": n_null,
        "blank": n_blank,
        "populated_pct": round(populated / max(ROW_COUNT, 1) * 100, 2),
    })

completeness_frame = pd.DataFrame(completeness).sort_values("populated_pct").reset_index(drop=True)
print(completeness_frame.to_string(index=False))
print()
empty_cols = completeness_frame[completeness_frame["populated_pct"] == 0]["column"].tolist()
print("columns with no data at all:", empty_cols if empty_cols else "none")

## 5. Cardinality and taxonomy detection

Every string column is checked for distinct count and for whether its values embed digits. A
low-cardinality string column whose values contain numbers is the pattern that produced the
NMTARA failure: it reads as a score and behaves as a label. Those columns are flagged so their
value lists get read rather than parsed.

In [ ]:
DIGIT_IN_TEXT = re.compile(r"[a-z].*\d|\d.*[a-z]", re.IGNORECASE)
RANGE_PATTERN = re.compile(r"\d\s*[-–]\s*\d")

card_rows = []
for c in STRING_COLS:
    d = src.select(SF.countDistinct(SF.col(f"`{c}`")).alias("d")).collect()[0]["d"]
    card_rows.append({"column": c, "distinct": int(d)})

cardinality_frame = pd.DataFrame(card_rows)
cardinality_frame["distinct_ratio"] = (
    cardinality_frame["distinct"] / max(ROW_COUNT, 1)).round(5)
cardinality_frame["shape"] = np.where(
    cardinality_frame["distinct"] <= LOW_CARDINALITY_MAX, "categorical",
    np.where(cardinality_frame["distinct_ratio"] > 0.5, "free_text_or_identifier", "high_cardinality"))
cardinality_frame = cardinality_frame.sort_values("distinct").reset_index(drop=True)
print(cardinality_frame.to_string(index=False))

CATEGORICAL_COLS = cardinality_frame[
    cardinality_frame["shape"] == "categorical"]["column"].tolist()

taxonomy_flags = []
value_lists = []
for c in CATEGORICAL_COLS:
    vc = (src.groupBy(SF.col(f"`{c}`").alias("value")).count()
          .orderBy(SF.desc("count")).limit(TOP_VALUES).toPandas())
    vc["column"] = c
    vc["pct"] = (vc["count"] / max(ROW_COUNT, 1) * 100).round(2)
    value_lists.append(vc[["column", "value", "count", "pct"]])
    vals = [str(v) for v in vc["value"].tolist() if v is not None]
    embeds_digits = sum(1 for v in vals if DIGIT_IN_TEXT.search(v))
    has_range = sum(1 for v in vals if RANGE_PATTERN.search(v))
    taxonomy_flags.append({
        "column": c,
        "distinct": int(cardinality_frame.loc[cardinality_frame["column"] == c, "distinct"].iloc[0]),
        "values_with_digits": embeds_digits,
        "values_with_ranges": has_range,
        "parse_risk": "HIGH" if has_range else ("MEDIUM" if embeds_digits else "low"),
    })

taxonomy_frame = pd.DataFrame(taxonomy_flags).sort_values(
    "parse_risk", ascending=False).reset_index(drop=True)
values_frame = pd.concat(value_lists, ignore_index=True) if value_lists else pd.DataFrame()

print()
print("Columns where a numeric parse would be unsafe")
print(taxonomy_frame.to_string(index=False))
print()
for c in taxonomy_frame[taxonomy_frame["parse_risk"] != "low"]["column"].tolist():
    print("-" * 90)
    print(c)
    print(values_frame[values_frame["column"] == c][["value", "count", "pct"]]
          .to_string(index=False))

## 6. Which column actually holds the justification

Name matching is a guess. This ranks every free-text candidate empirically: how often it is
populated, how long it runs, and how often it contains vocabulary from the medical necessity
knowledge base. The column that wins on concept hit rate is the clinical justification,
whatever it happens to be called.

In [ ]:
with open(KNOWLEDGE_PATH, "r") as f:
    KB = json.load(f)

ALL_TERMS = []
for c in KB["concepts"]:
    ALL_TERMS.extend(c["terms"])
for e in KB["bed_confinement"]["elements"]:
    ALL_TERMS.extend(e["terms"])
for e in KB["exclusions"]:
    ALL_TERMS.extend(e["terms"])
ALL_TERMS = sorted(set(ALL_TERMS))

TERM_RX = re.compile(r"(?<![a-z0-9])(?:" +
                     "|".join(re.escape(t) for t in sorted(ALL_TERMS, key=len, reverse=True)) +
                     r")(?![a-z0-9])")
VAGUE_RX = re.compile(r"(?<![a-z0-9])(?:" +
                      "|".join(re.escape(t) for t in
                               sorted(KB["non_specific_phrases"], key=len, reverse=True)) +
                      r")(?![a-z0-9])")

TEXT_CANDIDATES = cardinality_frame[
    cardinality_frame["shape"].isin(["free_text_or_identifier", "high_cardinality"])
]["column"].tolist()

sample = src.sample(withReplacement=False,
                    fraction=min(1.0, SAMPLE_N / max(ROW_COUNT, 1)),
                    seed=SAMPLE_SEED).limit(SAMPLE_N)
sample_pd = sample.toPandas()
print("profiling sample rows:", len(sample_pd))

cand_rows = []
for c in TEXT_CANDIDATES:
    s = sample_pd[c].astype(str).fillna("")
    s = s.replace({"None": "", "nan": ""})
    low = s.str.lower()
    populated = (s.str.strip().str.len() > 0)
    usable = (s.str.strip().str.len() >= MIN_TEXT_CHARS)
    hits = low.apply(lambda t: bool(TERM_RX.search(t))) if populated.any() else pd.Series(False, index=s.index)
    cand_rows.append({
        "column": c,
        "populated_pct": round(populated.mean() * 100, 1),
        "usable_pct": round(usable.mean() * 100, 1),
        "median_chars": int(s.str.len().median()),
        "p90_chars": int(s.str.len().quantile(0.90)),
        "concept_hit_pct": round(hits.mean() * 100, 1),
    })

candidate_frame = (pd.DataFrame(cand_rows)
                   .sort_values(["concept_hit_pct", "usable_pct"], ascending=False)
                   .reset_index(drop=True))
print()
print(candidate_frame.head(20).to_string(index=False))
print()
best = candidate_frame.iloc[0]["column"] if len(candidate_frame) else None
print("highest concept hit rate:", best)
print("currently mapped as notes:", COLS.get("notes"))
if best and COLS.get("notes") and best != COLS["notes"]:
    print("MISMATCH. The name-based mapping and the empirical ranking disagree. "
          "Read both columns before continuing.")

## 7. Grain and duplicates

Establishes what one row represents. If the order identifier repeats, the table is at leg or
event grain and every rate computed per row is weighted toward multi-leg trips.

In [ ]:
OID = COLS.get("order_id")
grain = {}
if OID:
    distinct_ids = src.select(SF.countDistinct(SF.col(f"`{OID}`"))).collect()[0][0]
    grain = {
        "rows": ROW_COUNT,
        "distinct_order_ids": int(distinct_ids),
        "rows_per_id": round(ROW_COUNT / max(int(distinct_ids), 1), 3),
    }
    dupes = (src.groupBy(SF.col(f"`{OID}`").alias("order_id")).count()
             .filter(SF.col("count") > 1).orderBy(SF.desc("count")).limit(10).toPandas())
    print(json.dumps(grain, indent=2))
    print()
    if len(dupes):
        print("identifiers appearing more than once, top 10")
        print(dupes.to_string(index=False))
        print()
        print("The table is not one row per order. Deduplicate or aggregate before rate work.")
    else:
        print("order identifier is unique. Table is one row per order.")

full_dupes = ROW_COUNT - src.dropDuplicates().count()
print()
print("fully duplicated rows:", f"{full_dupes:,}")
grain["fully_duplicated_rows"] = int(full_dupes)
grain_frame = pd.DataFrame([grain])

## 8. Date coverage

Confirms the window the analysis actually covers and whether volume is stable across it. A
month with a fraction of the expected volume usually means a partial load rather than a real
decline.

In [ ]:
DTE = COLS.get("created")
month_frame = pd.DataFrame()
if DTE:
    bounds = src.agg(SF.min(SF.col(f"`{DTE}`")).alias("min_dt"),
                     SF.max(SF.col(f"`{DTE}`")).alias("max_dt")).toPandas()
    print("date column:", DTE)
    print(bounds.to_string(index=False))
    print()
    month_frame = (src.withColumn("month", SF.date_format(SF.col(f"`{DTE}`"), "yyyy-MM"))
                   .groupBy("month").count().orderBy("month").toPandas())
    month_frame["pct"] = (month_frame["count"] / max(ROW_COUNT, 1) * 100).round(2)
    print(month_frame.to_string(index=False))
    print()
    nulls = int(src.filter(SF.col(f"`{DTE}`").isNull()).count())
    print("rows with no date:", f"{nulls:,}")
    if len(month_frame) > 2:
        med = month_frame["count"].median()
        thin = month_frame[month_frame["count"] < med * 0.5]
        print("months below half the median volume:",
              thin["month"].tolist() if len(thin) else "none")
else:
    print("no date column resolved")

## 9. Scope funnel

Each filter is applied in sequence and the rows lost at each step are recorded. The at-risk
denominator for any downstream rate is the last line of this table, not the table row count.

In [ ]:
SVC = COLS.get("service_level")
NOTES_COL = COLS.get("notes")

funnel = [{"step": "all rows in table", "rows": ROW_COUNT, "dropped": 0}]
scoped = src
prev = ROW_COUNT

if SVC:
    for code in EXCLUDED_SERVICE_CODES:
        scoped = scoped.filter(~SF.upper(SF.col(f"`{SVC}`")).contains(code))
        n = scoped.count()
        funnel.append({"step": f"exclude service code {code}", "rows": n, "dropped": prev - n})
        prev = n
else:
    funnel.append({"step": "service code filters SKIPPED, column unresolved",
                   "rows": prev, "dropped": 0})

if NOTES_COL:
    scoped = scoped.filter(SF.col(f"`{NOTES_COL}`").isNotNull())
    n = scoped.count()
    funnel.append({"step": "justification not null", "rows": n, "dropped": prev - n})
    prev = n

    scoped = scoped.filter(SF.length(SF.trim(SF.col(f"`{NOTES_COL}`"))) >= MIN_TEXT_CHARS)
    n = scoped.count()
    funnel.append({"step": f"justification at least {MIN_TEXT_CHARS} characters",
                   "rows": n, "dropped": prev - n})
    prev = n

funnel_frame = pd.DataFrame(funnel)
funnel_frame["pct_of_table"] = (funnel_frame["rows"] / max(ROW_COUNT, 1) * 100).round(2)
print(funnel_frame.to_string(index=False))
print()
IN_SCOPE = prev
print("orders in scope for medical necessity review:", f"{IN_SCOPE:,}",
      f"({IN_SCOPE / max(ROW_COUNT, 1) * 100:.1f}% of the table)")

## 10. Justification text profile

Length distribution, and how much of the text is boilerplate. A justification string that
repeats verbatim across hundreds of orders is a template default, not documentation of a
patient's condition, and it will score identically every time.

In [ ]:
text_profile = pd.DataFrame()
repeated_frame = pd.DataFrame()
if NOTES_COL:
    txt = sample_pd[NOTES_COL].astype(str).replace({"None": "", "nan": ""}).fillna("")
    lengths = txt.str.strip().str.len()
    text_profile = pd.DataFrame([{
        "metric": m, "value": v} for m, v in [
        ("rows sampled", len(txt)),
        ("blank", int((lengths == 0).sum())),
        ("under 20 chars", int(((lengths > 0) & (lengths < MIN_TEXT_CHARS)).sum())),
        ("usable", int((lengths >= MIN_TEXT_CHARS).sum())),
        ("median chars", int(lengths.median())),
        ("p25 chars", int(lengths.quantile(0.25))),
        ("p75 chars", int(lengths.quantile(0.75))),
        ("p95 chars", int(lengths.quantile(0.95))),
        ("max chars", int(lengths.max())),
    ]])
    print(text_profile.to_string(index=False))
    print()

    norm = txt.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
    vc = norm[norm.str.len() >= MIN_TEXT_CHARS].value_counts()
    repeated = vc[vc > 1]
    print("distinct justification strings:", f"{int(vc.size):,}")
    print("strings appearing more than once:", f"{int(repeated.size):,}")
    print("share of usable rows covered by repeated strings:",
          f"{repeated.sum() / max(vc.sum(), 1) * 100:.1f}%")
    print()
    repeated_frame = (repeated.head(15).rename("orders").reset_index()
                      .rename(columns={"index": "justification_text"}))
    repeated_frame["chars"] = repeated_frame["justification_text"].str.len()
    if INCLUDE_TEXT_SAMPLES:
        print(repeated_frame.to_string(index=False))
    else:
        print(repeated_frame[["orders", "chars"]].to_string(index=False))
        print("set INCLUDE_TEXT_SAMPLES to see the strings themselves")

    vague_only = norm.apply(lambda t: bool(VAGUE_RX.search(t)) and not bool(TERM_RX.search(t)))
    print()
    print("rows whose justification is only non-specific phrasing:",
          f"{int(vague_only.sum()):,} ({vague_only.mean() * 100:.1f}%)")

## 11. Knowledge base coverage

How much of the criteria vocabulary the data actually exercises. A concept with a hit rate near
zero either does not occur in this population or is written a different way here than the
knowledge file expects, and the two cases are distinguished by reading the notes, not by the
rate alone.

In [ ]:
coverage_frame = pd.DataFrame()
if NOTES_COL:
    low_txt = sample_pd[NOTES_COL].astype(str).replace({"None": "", "nan": ""}).fillna("").str.lower()

    def rx(terms):
        return re.compile(r"(?<![a-z0-9])(?:" +
                          "|".join(re.escape(t) for t in sorted(terms, key=len, reverse=True)) +
                          r")(?![a-z0-9])")

    rows = []
    for c in KB["concepts"]:
        hit = low_txt.str.contains(rx(c["terms"]), regex=True)
        rows.append({"kind": "concept", "name": c["concept"], "axis": c["axis"],
                     "status": c["status"], "source_id": c["source_id"],
                     "orders": int(hit.sum()), "pct": round(hit.mean() * 100, 2)})
    for e in KB["bed_confinement"]["elements"]:
        hit = low_txt.str.contains(rx(e["terms"]), regex=True)
        rows.append({"kind": "bed_element", "name": e["element"], "axis": "mobility",
                     "status": "cited", "source_id": "FACILITY_TRAINING",
                     "orders": int(hit.sum()), "pct": round(hit.mean() * 100, 2)})
    for e in KB["exclusions"]:
        hit = low_txt.str.contains(rx(e["terms"]), regex=True)
        rows.append({"kind": "exclusion", "name": e["exclusion_id"], "axis": "",
                     "status": "", "source_id": e["source_id"],
                     "orders": int(hit.sum()), "pct": round(hit.mean() * 100, 2)})

    coverage_frame = pd.DataFrame(rows).sort_values(
        ["kind", "pct"], ascending=[True, False]).reset_index(drop=True)
    print(coverage_frame.to_string(index=False))
    print()

    any_hit = low_txt.str.contains(TERM_RX, regex=True)
    print("orders matching at least one knowledge base term:",
          f"{int(any_hit.sum()):,} ({any_hit.mean() * 100:.1f}%)")
    print()
    dead = coverage_frame[coverage_frame["orders"] == 0]["name"].tolist()
    print("vocabulary that never fires in this sample:", dead if dead else "none")

## 12. Facility and payer

Documentation quality is not uniform across the pilot sites. This is the split the business
case rests on, so it is measured directly rather than assumed.

In [ ]:
facility_frame = pd.DataFrame()
payer_frame = pd.DataFrame()
FAC = COLS.get("facility")
PAY = COLS.get("payer")

if FAC and NOTES_COL:
    tmp = sample_pd[[FAC, NOTES_COL]].copy()
    tmp.columns = ["facility", "notes"]
    tmp["notes"] = tmp["notes"].astype(str).replace({"None": "", "nan": ""}).fillna("")
    tmp["usable"] = tmp["notes"].str.strip().str.len() >= MIN_TEXT_CHARS
    tmp["concept_hit"] = tmp["notes"].str.lower().str.contains(TERM_RX, regex=True)
    tmp["chars"] = tmp["notes"].str.len()
    facility_frame = (tmp.groupby("facility")
                      .agg(orders=("notes", "size"),
                           usable_pct=("usable", lambda s: round(s.mean() * 100, 1)),
                           concept_hit_pct=("concept_hit", lambda s: round(s.mean() * 100, 1)),
                           median_chars=("chars", lambda s: int(s.median())))
                      .sort_values("orders", ascending=False).reset_index())
    print(facility_frame.head(25).to_string(index=False))

if PAY:
    payer_frame = (sample_pd.groupby(PAY).size().rename("orders")
                   .sort_values(ascending=False).head(20).reset_index())
    payer_frame.columns = ["payer", "orders"]
    print()
    print(payer_frame.to_string(index=False))

## 13. Red flags

Conditions that would invalidate downstream work if left unresolved. An empty table here is
the result to hope for.

In [ ]:
flags = []

for role, col in COLS.items():
    if col is None:
        flags.append({"severity": "blocking", "check": f"role {role} unresolved",
                      "detail": "Set it in OVERRIDES in section 3."})

if len(taxonomy_frame):
    for r in taxonomy_frame[taxonomy_frame["parse_risk"] == "HIGH"].to_dict("records"):
        flags.append({"severity": "blocking", "check": f"{r['column']} holds numeric ranges",
                      "detail": "Values embed ranges such as 1-5. Do not parse a level from it. "
                                "Map the labels explicitly."})
    for r in taxonomy_frame[taxonomy_frame["parse_risk"] == "MEDIUM"].to_dict("records"):
        flags.append({"severity": "review", "check": f"{r['column']} embeds digits in labels",
                      "detail": "Read the value list before deriving anything numeric from it."})

if grain.get("rows_per_id", 1) > 1.0:
    flags.append({"severity": "blocking", "check": "order identifier is not unique",
                  "detail": f"{grain['rows_per_id']} rows per identifier. Table is not at "
                            "order grain."})

if grain.get("fully_duplicated_rows", 0) > 0:
    flags.append({"severity": "review", "check": "fully duplicated rows present",
                  "detail": f"{grain['fully_duplicated_rows']:,} rows are exact duplicates."})

if len(candidate_frame) and COLS.get("notes") and candidate_frame.iloc[0]["column"] != COLS["notes"]:
    flags.append({"severity": "blocking", "check": "justification column disputed",
                  "detail": f"Name matching chose {COLS['notes']}, concept hit rate favours "
                            f"{candidate_frame.iloc[0]['column']}."})

if len(empty_cols):
    flags.append({"severity": "review", "check": "columns with no data",
                  "detail": ", ".join(empty_cols[:10])})

if IN_SCOPE / max(ROW_COUNT, 1) < 0.25:
    flags.append({"severity": "review", "check": "scope filters remove most of the table",
                  "detail": f"{IN_SCOPE:,} of {ROW_COUNT:,} rows survive. Confirm the filters "
                            "are correct before quoting any rate."})

if len(coverage_frame):
    dead_terms = coverage_frame[coverage_frame["orders"] == 0]
    if len(dead_terms):
        flags.append({"severity": "review", "check": "knowledge base vocabulary never fires",
                      "detail": ", ".join(dead_terms["name"].tolist()[:12])})

flags_frame = pd.DataFrame(flags) if flags else pd.DataFrame(
    [{"severity": "none", "check": "no red flags raised", "detail": ""}])
print(flags_frame.to_string(index=False))

## 14. Output

Aggregate profiles only. Raw justification text is written to the workbook only when
`INCLUDE_TEXT_SAMPLES` is set, because the source table carries patient identifiers and the
workbook is shared.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def sanitize(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v) if isinstance(v, (np.floating, float)) else int(v)
    return CONTROL_CHARS.sub(" ", str(v))[:32000]


repeated_out = repeated_frame if INCLUDE_TEXT_SAMPLES else (
    repeated_frame[["orders", "chars"]] if len(repeated_frame) else repeated_frame)

TABS = [
    ("Red Flags", flags_frame),
    ("Schema", schema_frame),
    ("Completeness", completeness_frame),
    ("Cardinality", cardinality_frame),
    ("Parse Risk", taxonomy_frame),
    ("Category Values", values_frame),
    ("Column Resolution", resolution_frame),
    ("Text Candidates", candidate_frame),
    ("Grain", grain_frame),
    ("Monthly Volume", month_frame),
    ("Scope Funnel", funnel_frame),
    ("Text Profile", text_profile),
    ("Repeated Text", repeated_out),
    ("KB Coverage", coverage_frame),
    ("By Facility", facility_frame),
    ("By Payer", payer_frame),
]

wb = Workbook()
wb.remove(wb.active)
for name, frame in TABS:
    ws = wb.create_sheet(name[:31])
    if frame is None or len(frame) == 0:
        ws.append(["no rows"])
        continue
    ws.append([sanitize(c) for c in frame.columns])
    for cell in ws[1]:
        cell.font = Font(name="Calibri", bold=True)
    for rec in frame.itertuples(index=False):
        ws.append([sanitize(v) for v in rec])
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for i, col in enumerate(frame.columns, start=1):
        width = 60 if str(col) in ("detail", "statement", "justification_text", "alternates") else 20
        ws.column_dimensions[get_column_letter(i)].width = width

xlsx_name = f"med_nec_data_validation_{RUN_TS}.xlsx"
local_xlsx = os.path.join(LOCAL_DIR, xlsx_name)
wb.save(local_xlsx)

try:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    shutil.copyfile(local_xlsx, os.path.join(OUTPUT_DIR, xlsx_name))
    print("wrote", os.path.join(OUTPUT_DIR, xlsx_name))
except Exception as e:
    print("workspace copy skipped:", e)
    print("local copy at", local_xlsx)

print()
print("rows in table:", f"{ROW_COUNT:,}")
print("orders in scope:", f"{IN_SCOPE:,}")
print("blocking flags:", int((flags_frame["severity"] == "blocking").sum()))
print("review flags:", int((flags_frame["severity"] == "review").sum()))

## 15. What to do with the result

Read the Red Flags tab first. A blocking flag means a number produced downstream will be wrong
rather than imprecise.

Order of resolution:

1. Any unresolved role. Set it in `OVERRIDES` in section 3 and re-run.
2. A disputed justification column. Read both candidates before choosing. The empirical ranking
   in section 6 is evidence, not a decision.
3. Any column flagged HIGH parse risk. Those values are labels. Map them explicitly, the way
   the nurse navigation breakout field had to be mapped, rather than extracting a number.
4. A non-unique order identifier. Every per-order rate is weighted until this is settled.

The scope funnel in section 9 gives the denominator for the medical necessity work. Quote that
number rather than the table row count.

The knowledge base coverage in section 11 tells you whether the criteria vocabulary matches how
these facilities actually write. Vocabulary that never fires is either a condition this
population does not have or a phrasing gap in `med_nec_knowledge.json`, and only reading a
sample of notes distinguishes the two.